In [48]:
# GOAL:
## Find staging/stacking orderings of Na single layers.
## I will use three Na layer structures for all orderings.
## Preparation: POSCAR file that have three layers of Na, and one layer has Na-Vac Orderings where Vac is marked by K.
## Approach: 1. Classify Na sites into layers based on z coordinate.
##           2. For each pair of layers, find all translation vectors (in fractional xy) that map one layer onto the other.
##           3. Collect all unique translation vectors across Na/K layer to Na layers.
##           4. Use these translation vectors to build up possible stacking sequences.
##           5. Deduplicate with StructureMatcher.
##           6. Use MACE to evaluate energies of candidates.
## Note: This is a preliminary exploration; final implementation will be in a .py script.
## Note: This code assumes that only one type of vacancy ordering exists in the layers.
## Note: This code asssmes No transition metal mixing in the transition metal layers.

In [49]:
from __future__ import annotations
from pathlib import Path
import json
import math
from itertools import product
from typing import List, Tuple, Dict

import numpy as np
from pymatgen.core import Structure
from pymatgen.io.vasp import Poscar

from pymatgen.analysis.structure_matcher import StructureMatcher

In [63]:
# --- I/O roots ---
DATA_ROOTS = [
    Path("../data/O3_orderings"),
    Path("../data/P3_orderings"),
]
NAME_PATTERNS = ["AJT*_w_vac.vasp", "JLK*_w_vac.vasp"]  # patterns to pick up; expand if needed
OUTPUT_ROOT = Path("../data/mace_screening/singleTM/ordering_staging")

# --- Tolerances ---
XY_MATCH_DECIMALS = 2          # rounding precision for hashing 2D fractional coords
XY_MATCH_TOL = 10**(-XY_MATCH_DECIMALS) * 2
Z_LAYER_GAP_MIN = 1e-3     

# Structure Matcher parameters
SM_LATTICE_TOL = 0.2 
SM_SITE_TOL    = 0.3
SM_ANGLE_TOL   = 5.0

In [64]:
def frac01(x: np.ndarray) -> np.ndarray:
    """Wrap fractional coordinates into [0,1)."""
    return x - np.floor(x)


def key2(xy: np.ndarray, decimals: int = XY_MATCH_DECIMALS) -> Tuple[int, int]:
    """Hashable key for 2D fractional coords under PBC (rounded)."""
    v = frac01(np.asarray(xy))[:2]
    return tuple(np.round(v, decimals=decimals).astype(float))


def split_three_layers_by_z(struct: Structure, elems=("Na", "K")) -> List[List[int]]:
    """
    Return indices (into struct.sites) for the three Na/K layers by clustering
    fractional z with a 'largest gaps' method (robust, no sklearn).
    """
    # Collect (z, idx) for Na or K only
    z_idx = []
    for i, site in enumerate(struct):
        sp = site.specie.symbol if hasattr(site, "specie") else site.species_string
        if sp in elems:
            z_idx.append((site.frac_coords[2] % 1.0, i))
    if len(z_idx) == 0:
        raise ValueError("No Na/K sites found.")
    # Sort by z and compute circular gaps to find 3 largest gaps (=> 3 clusters)
    z_idx.sort(key=lambda t: t[0])
    zs = [z for z, _ in z_idx]
    # Circular list of gaps
    gaps = []
    for a, b in zip(zs, zs[1:]):
        gaps.append(b - a)
    gaps.append((zs[0] + 1.0) - zs[-1])  # wrap-around
    # Pick the 3 largest gaps; their endpoints are cut points
    # (For 3 layers we expect exactly 3 clusters.)
    cut_positions = sorted(np.argsort(gaps)[-3:])
    # Build clusters splitting at those cuts
    # We rotate the list so the first cut becomes the end
    rotate = (cut_positions[0] + 1) % len(z_idx)
    z_idx_rot = z_idx[rotate:] + z_idx[:rotate]
    gaps_rot = gaps[rotate:] + gaps[:rotate]
    cut_positions_rot = [(c - rotate) % len(gaps) for c in cut_positions]
    cut_positions_rot = sorted(cut_positions_rot)

    clusters: List[List[int]] = []
    start = 0
    for cp in cut_positions_rot:
        # slice from start..cp (inclusive of start, exclusive of cp+1)
        # but we add +1 because cuts are between elements
        end = cp + 1
        chunk = z_idx_rot[start:end]
        if chunk:
            clusters.append([idx for _, idx in chunk])
        start = end
    if start < len(z_idx_rot):
        clusters.append([idx for _, idx in z_idx_rot[start:]])

    # Basic sanity: exactly 3 clusters
    if len(clusters) != 3:
        raise RuntimeError(f"Expected 3 Na/K layers, found {len(clusters)}")

    return clusters


def layer_xy_sets(struct: Structure, indices: List[int]) -> Tuple[List[int], np.ndarray, Dict[Tuple[float,float], List[int]]]:
    """
    For a given layer (site indices), return:
      - the same indices,
      - Nx2 array of fractional (x,y),
      - dict mapping hashed xy key -> list of site indices 
    """
    xys = []
    mapping = {}
    for i in indices:
        xy = frac01(struct[i].frac_coords)[:2]
        xys.append(xy)
        mapping.setdefault(key2(xy), []).append(i)
    return indices, np.array(xys), mapping


def translations_that_map(P: np.ndarray, Q: np.ndarray) -> List[np.ndarray]:
    """
    Find all translation vectors v (2D fractional) such that 
    (P + v) % 1 is a permutation of Q within rounding tolerance.
    Strategy: Try v = q - p0 for every q in Q using an arbitrary p0 in P, then verify full-set membership.
    """
    if len(P) != len(Q):
        raise ValueError("Layer site counts differ; can't define a bijective translation mapping.")

    if len(P) == 0:
        return []

    p0 = P[0]
    Q_keys = {key2(q) for q in Q}
    Vs = []
    seen = set()

    for q in Q:
        v = frac01(q - p0)       # candidate translation
        # Verify mapping: every p in P goes to some q in Q under +v
        ok = True
        for p in P:
            cand = frac01(p + v)
            if key2(cand) not in Q_keys:
                ok = False
                break
        if ok:
            k = key2(v)
            if k not in seen:
                seen.add(k)
                Vs.append(v)
    return Vs


def k_mask_in_layer(struct: Structure, layer_indices: List[int]) -> Tuple[np.ndarray, List[int]]:
    """Return Nx2 fractional xy for K sites in the given layer + their site indices."""
    k_xy = []
    k_idx = []
    for i in layer_indices:
        sp = struct[i].specie.symbol if hasattr(struct[i], "specie") else struct[i].species_string
        if sp == "K":
            k_idx.append(i)
            k_xy.append(frac01(struct[i].frac_coords)[:2])
    return np.array(k_xy), k_idx


def apply_pattern_to_layer_by_translation(
    struct,
    P_xy,
    pattern,
    target_xy_map,
    v2d,
):
    """
    Apply Na/K pattern from template layer to a target layer using translation v2d.

    Inputs:
      - struct: pymatgen Structure to modify in-place
      - P_xy: (N,2) fractional (x,y) of template Na/K sites (from layer_xy_sets)
      - pattern: dict[(xy_key)] -> "Na" or "K" for template layer
      - target_xy_map: dict[(xy_key)] -> [site indices] for target layer (from layer_xy_sets)
      - v2d: 2D fractional translation vector (from translations_that_map)

    For each template site at xy with species s:
      1. xy' = (xy + v2d) % 1
      2. find unique site index in target_xy_map[xy']
      3. set that site’s species to s

    Assumes v2d is a valid mapping of the Na sublattice (checked beforehand).
    """
    for xy in P_xy:
        tpl_key = key2(xy)
        s = pattern[tpl_key]  # Check specie "Na" or "K"

        xy_shift = frac01(xy + v2d)
        tgt_key = key2(xy_shift) # Key in target layer

        if tgt_key not in target_xy_map:
            raise RuntimeError(
                f"Invalid translation: no target site at translated position {xy_shift}."
            )

        idxs = target_xy_map[tgt_key] # Index in target layer
        if len(idxs) != 1:
            raise RuntimeError(
                f"Ambiguous mapping at {xy_shift}: multiple target sites {idxs}."
            )

        idx = idxs[0]
        site = struct[idx]
        struct[idx] = (s, site.frac_coords)


def process_one_poscar(poscar_path: Path, out_root: Path) -> None:
    struct = Structure.from_file(str(poscar_path))

    # 1) Find the three Na/K layers
    layers = split_three_layers_by_z(struct, elems=("Na", "K"))  # uses existing helper

    # 2) Identify template layer: the one containing at least one K (Na/K ordering)
    k_counts = []
    for L in layers:
        k_in_L = sum(
            (struct[i].specie.symbol if hasattr(struct[i], "specie") else struct[i].species_string) == "K"
            for i in L
        )
        k_counts.append(k_in_L)

    try:
        template_idx = k_counts.index(next(c for c in k_counts if c > 0))
    except StopIteration:
        raise RuntimeError(f"{poscar_path.name}: No K found in any Na layer; cannot define template.")

    template_layer = layers[template_idx]
    other_layers_idx = [i for i in range(3) if i != template_idx]
    L2, L3 = layers[other_layers_idx[0]], layers[other_layers_idx[1]]

    # 3) Use existing layer_xy_sets to get coords + maps
    tmpl_indices, P_xy, P_map = layer_xy_sets(struct, template_layer)
    _, Q2_xy, Q2_map = layer_xy_sets(struct, L2)
    _, Q3_xy, Q3_map = layer_xy_sets(struct, L3)

    # Build Na/K pattern on template layer from P_map (no new scan logic)
    pattern = {}
    for key, idxs in P_map.items():
        if len(idxs) != 1:
            raise RuntimeError(
                f"{poscar_path.name}: Multiple sites at same (x,y) in template layer: {idxs}"
            )
        idx = idxs[0]
        sp = struct[idx].specie.symbol if hasattr(struct[idx], "specie") else struct[idx].species_string
        if sp not in ("Na", "K"):
            raise RuntimeError(
                f"{poscar_path.name}: Non-Na/K species {sp} in template Na layer at site {idx}"
            )
        pattern[key] = sp

    # Basic sanity: all three layers must share same Na/K site count
    if not (len(P_xy) == len(Q2_xy) == len(Q3_xy)):
        raise RuntimeError(
            f"{poscar_path.name}: Na/K site counts differ between layers: "
            f"{len(P_xy)}, {len(Q2_xy)}, {len(Q3_xy)}."
        )

    # 4) Get pure translations from template layer → other layers using existing translations_that_map
    V2 = translations_that_map(P_xy, Q2_xy)
    V3 = translations_that_map(P_xy, Q3_xy)

    if not V2 or not V3:
        raise RuntimeError(
            f"{poscar_path.name}: No valid translational mapping(s) found for Na sublattice."
        )

    # 5) Enumerate all (t2, t3) combinations, applying ONLY translations of the pattern
    stem = poscar_path.stem
    phase_dir = poscar_path.parent.name
    base_out = out_root / phase_dir / stem
    base_out.mkdir(parents=True, exist_ok=True)

    candidates_structs = []
    candidates_meta = []
    total_candidates = 0

    for i2, v2 in enumerate(V2):
        for i3, v3 in enumerate(V3):
            new_struct = struct.copy()

            # Apply template Na/K pattern to layers 2 and 3 via translations
            apply_pattern_to_layer_by_translation(new_struct, P_xy, pattern, Q2_map, v2)
            apply_pattern_to_layer_by_translation(new_struct, P_xy, pattern, Q3_map, v3)

            new_struct = new_struct.get_sorted_structure()

            meta = {
                "source_poscar": str(poscar_path),
                "phase_dir": phase_dir,
                "template_layer_index": template_idx,
                "other_layers_index_order": other_layers_idx,
                "t2_index": i2,
                "t3_index": i3,
                "t2_vector_frac_xy": [float(v2[0]), float(v2[1])],
                "t3_vector_frac_xy": [float(v3[0]), float(v3[1])],
                "k_count_per_layer": k_counts,
                "layer_site_count": int(len(P_xy)),
            }

            candidates_structs.append(new_struct)
            candidates_meta.append(meta)
            total_candidates += 1

    # 6) Deduplicate by symmetry, treating Na/K as distinct
    sm = StructureMatcher(
        ltol=SM_LATTICE_TOL,
        stol=SM_SITE_TOL,
        angle_tol=SM_ANGLE_TOL,
        primitive_cell=False,
        scale=True,
        attempt_supercell=False,
    )

    groups = sm.group_structures(candidates_structs)
    idx_map = {id(s): i for i, s in enumerate(candidates_structs)}

    kept = 0
    for g_idx, group in enumerate(groups):
        rep_struct = group[0]
        rep_i = idx_map[id(rep_struct)]
        rep_meta = candidates_meta[rep_i]

        out_dir = base_out / f"group_{g_idx:03d}"
        out_dir.mkdir(parents=True, exist_ok=True)
        Poscar(rep_struct).write_file(out_dir / "POSCAR")

        members = []
        for member in group:
            mi = idx_map[id(member)]
            mm = candidates_meta[mi]
            members.append({
                "t2_index": mm["t2_index"],
                "t3_index": mm["t3_index"],
                "t2_vector_frac_xy": mm["t2_vector_frac_xy"],
                "t3_vector_frac_xy": mm["t3_vector_frac_xy"],
            })

        group_meta = {
            "representative_from": {
                "t2_index": rep_meta["t2_index"],
                "t3_index": rep_meta["t3_index"],
                "t2_vector_frac_xy": rep_meta["t2_vector_frac_xy"],
                "t3_vector_frac_xy": rep_meta["t3_vector_frac_xy"],
            },
            "members": members,
            "counts": {
                "|V2|": len(V2),
                "|V3|": len(V3),
                "candidates": total_candidates,
                "unique_groups": len(groups),
            },
            "matcher_tolerances": {
                "ltol": SM_LATTICE_TOL,
                "stol": SM_SITE_TOL,
                "angle_tol": SM_ANGLE_TOL,
            },
            "notes": (
                "All three layers share an identical Na/K pattern up to in-plane translation. "
                "Na and K treated as distinct during symmetry matching."
            ),
        }

        with open(out_dir / "group_meta.json", "w") as f:
            json.dump(group_meta, f, indent=2)

        kept += 1

    print(
        f"[{phase_dir}/{stem}] {total_candidates} translation-derived candidates "
        f"→ {kept} symmetry-unique patterns."
    )


In [65]:
# Before run All, collect all Orderings.
def collect_poscars(roots, patterns):
    seen = set()
    files = []
    for root in roots:
        if not root.exists():
            print(f"Warning: {root} does not exist; skipping.")
            continue
        for pat in patterns:
            for p in root.rglob(pat):
                if p.is_file():
                    sp = str(p)
                    if sp not in seen:
                        seen.add(sp)
                        files.append(p)
    files.sort()
    return files

poscars = collect_poscars(DATA_ROOTS, NAME_PATTERNS)
print(f"Found {len(poscars)} POSCARs:")
for p in poscars:
    print("  -", p)

Found 27 POSCARs:
  - ../data/O3_orderings/AJT_a_w_vac.vasp
  - ../data/O3_orderings/AJT_b_w_vac.vasp
  - ../data/O3_orderings/AJT_c_w_vac.vasp
  - ../data/O3_orderings/AJT_d_w_vac.vasp
  - ../data/O3_orderings/AJT_e_w_vac.vasp
  - ../data/O3_orderings/AJT_f_w_vac.vasp
  - ../data/O3_orderings/AJT_g_w_vac.vasp
  - ../data/O3_orderings/AJT_h_w_vac.vasp
  - ../data/O3_orderings/AJT_i_w_vac.vasp
  - ../data/O3_orderings/AJT_k_w_vac.vasp
  - ../data/O3_orderings/AJT_l_w_vac.vasp
  - ../data/O3_orderings/AJT_m_w_vac.vasp
  - ../data/O3_orderings/AJT_n_w_vac.vasp
  - ../data/O3_orderings/AJT_o_w_vac.vasp
  - ../data/O3_orderings/AJT_p_w_vac.vasp
  - ../data/O3_orderings/AJT_q_w_vac.vasp
  - ../data/O3_orderings/AJT_r_w_vac.vasp
  - ../data/O3_orderings/JLK_b_w_vac.vasp
  - ../data/O3_orderings/JLK_c_w_vac.vasp
  - ../data/O3_orderings/JLK_theta_4_w_vac.vasp
  - ../data/O3_orderings/JLK_theta_5_w_vac.vasp
  - ../data/P3_orderings/JLK_delta_w_vac.vasp
  - ../data/P3_orderings/JLK_eta_w_vac.vas

In [66]:
ok = 0
for i, p in enumerate(poscars, 1):
    try:
        print(f"\n[{i}/{len(poscars)}] Processing: {p}")
        process_one_poscar(p, OUTPUT_ROOT)
        ok += 1
    except Exception as e:
        print(f"ERROR processing {p} → {e}")

print(f"\nDone. Successfully processed {ok}/{len(poscars)} files.")
print(f"Outputs under: {OUTPUT_ROOT.resolve()}")


[1/27] Processing: ../data/O3_orderings/AJT_a_w_vac.vasp
[O3_orderings/AJT_a_w_vac] 49 translation-derived candidates → 6 symmetry-unique patterns.

[2/27] Processing: ../data/O3_orderings/AJT_b_w_vac.vasp
[O3_orderings/AJT_b_w_vac] 16 translation-derived candidates → 4 symmetry-unique patterns.

[3/27] Processing: ../data/O3_orderings/AJT_c_w_vac.vasp
[O3_orderings/AJT_c_w_vac] 16 translation-derived candidates → 3 symmetry-unique patterns.

[4/27] Processing: ../data/O3_orderings/AJT_d_w_vac.vasp
[O3_orderings/AJT_d_w_vac] 9 translation-derived candidates → 2 symmetry-unique patterns.

[5/27] Processing: ../data/O3_orderings/AJT_e_w_vac.vasp
[O3_orderings/AJT_e_w_vac] 36 translation-derived candidates → 7 symmetry-unique patterns.

[6/27] Processing: ../data/O3_orderings/AJT_f_w_vac.vasp
[O3_orderings/AJT_f_w_vac] 25 translation-derived candidates → 5 symmetry-unique patterns.

[7/27] Processing: ../data/O3_orderings/AJT_g_w_vac.vasp
[O3_orderings/AJT_g_w_vac] 16 translation-derived

In [2]:
from pathlib import Path

In [3]:
test = Path('../data/O3_orderings/AJT_a_w_vac.vasp')

In [4]:
test.stem

'AJT_a_w_vac'